In [100]:
import pandas as pd
import networkx as nx
from collections import defaultdict


In [104]:
relationships = pd.read_csv("data/hero_relationships.csv", keep_default_na=False).drop(columns=["Unnamed: 7", "Global Notes"])
hero_tier_list = pd.read_csv("data/hero_tier_list.csv").drop(columns=["Unnamed: 3"])
all_heroes = relationships["name"].to_list()
hero_tier_list[hero_tier_list["Name"] == "Frank"]

,Name,Position,Tier
50,Frank,Mid,A
98,Frank,Top,S


In [96]:
# Numeric map of tiers 
tier_map = {
    "S": 5,
    "A": 4,
    "B": 3,
    "C": 2,
    "D": 1
}

hero_tier_list["tier_score"] = hero_tier_list["Tier"].map(tier_map)
heros = hero_tier_list["Name"].unique().tolist()
print(heros[:5])

['Ada', 'BaJie', 'Bariel', 'Bond', 'Bunu Shan']


In [97]:
def get_hero_best_positions(hero_name: str | list[str], tier=1, verbose=False) -> dict[str, list[dict]]:
    output = {}

    names = [hero_name] if isinstance(hero_name, str) else hero_name.copy()

    for name in names:
        hero_rows = hero_tier_list[hero_tier_list["Name"] == name]
        max_score = hero_rows["tier_score"].max()
        target_score = max_score - (tier - 1)

        best_positions = hero_rows.loc[
            hero_rows["tier_score"] == target_score,
            ["Position", "Tier"]
        ].to_dict("records")

        if not best_positions:
            print(f"No heroes found under the name {name}.")
            continue

        position_names = [position["Position"] for position in best_positions]
        print_out = f"{name}'s best position is {', '.join(position_names)}."

        if verbose:
            print(print_out)

        output[name] = best_positions

    return output

#best_positions = get_hero_best_positions(["Ada", "Felyn", "Foso", "Frank"], verbose=True)
best_positions = get_hero_best_positions("Frank", verbose=True, tier=1)
print(best_positions)

Frank's best position is Top.
{'Frank': [{'Position': 'Top', 'Tier': 'S'}]}


In [98]:
# Get the best heroes for a position
def get_position_best_heroes(position: str | list[str], tier=1, verbose=False) -> dict[str, list[dict]]:
    output = {}

    positions = [position] if isinstance(position, str) else position.copy()

    for position in positions:
        position_rows = hero_tier_list[hero_tier_list["Position"] == position]
        max_score = position_rows["tier_score"].max()
        target_score = max_score - (tier - 1)

        best_heroes = position_rows.loc[
            position_rows["tier_score"] == target_score,
            ["Name", "Tier"]
        ].to_dict("records")

        if not best_heroes:
            print(f"No heroes found for the {position} position.")
            continue

        hero_names = [hero["Name"] for hero in best_heroes]
        print_out = f"Best heroes for the {position.lower()} position are {', '.join(hero_names)}."

        if verbose:
            print(print_out)

        output[position] = best_heroes

    return output

get_position_best_heroes(["Top", "Mid"], verbose=True)

Best heroes for the top position are Frank, Kid, Wukong, Foso.
Best heroes for the mid position are Aurelio, Merisi, Wolfgang, Foso.


{'Top': [{'Name': 'Frank', 'Tier': 'S'},
  {'Name': 'Kid', 'Tier': 'S'},
  {'Name': 'Wukong', 'Tier': 'S'},
  {'Name': 'Foso', 'Tier': 'S'}],
 'Mid': [{'Name': 'Aurelio', 'Tier': 'S'},
  {'Name': 'Merisi', 'Tier': 'S'},
  {'Name': 'Wolfgang', 'Tier': 'S'},
  {'Name': 'Foso', 'Tier': 'S'}]}

In [99]:
# Confirm heroes in relationships are == to heroes in tiers 
heroes_t = set(hero_tier_list["Name"].unique().tolist())
all_heroes = set(all_heroes)


mismatches = []
for hero in heroes_t:
    if hero in all_heroes:
        continue
    mismatches.append((hero))

if (len(heroes_t) != len(all_heroes)) or mismatches:
    print("Names do not match.")
else:
    print("Names match.")

Names match.


In [ ]:
# Build tag -> hero lookup
lookup = defaultdict(list)
for hero in all_heroes:
    hero_data = relationships[relationships["name"] == hero]
    hero_tags = hero_data["tags"].tolist()[0].split(",")
    for tag in hero_tags:
        lookup[tag].append(hero)
print(lookup)

defaultdict(<class 'list'>, {'True Damage': ['Manta', 'Mihawk', 'Digo'], 'Assassin': ['Manta', 'Lubos', 'Xiangxi Ke', 'Lan', 'Gillis', 'Raven'], 'Crit': ['Manta', 'BaJie', 'Bariel', 'Raven', 'Quinn'], 'Targeting': ['Lubos', 'Xiangxi Ke', 'Leon'], 'Tank': ['Peter', 'Miki', 'Merisi', 'Big Foot', 'Cubey', 'Frank', 'Bart', 'Gillis', 'Matata', 'Fatty White'], 'Taunt': ['Peter'], 'Shield': ['Miki', 'Qin Hu', 'Palulu'], 'Lane Bully': ['Merisi', 'Felyn', 'Wolfgang', 'Qube'], 'Debuff': ['Big Foot', 'Crank', 'Felicity', 'Lady Deadfire', 'Shougong Lei', 'Tiger Boy'], 'Summon': ['Qin Hu', 'Kaka', 'Omaha'], 'Squishy': ['Niels', 'Felyn', 'Enidi'], 'Skirmisher': ['Zealot', 'Aurelio', 'Bond', 'Babe', 'Lan', 'Raven', 'Ada', 'Anna'], 'Multi-Attacker': ['Zealot', 'Kamaitachi', 'Mo', 'Elemi'], 'Initiator': ['Aurelio', 'Reinhardt', 'Hakuna'], 'Fighter': ['Aurelio', 'Mo'], 'ADC': ['Bond', 'Deep Space', 'Ada', 'Elemi', 'Omaha', 'Quinn'], 'Search': ['Blocker', 'Beverly'], 'Round 1 Damage': ['Reinhardt', 'Haku

In [101]:
# Build hero role list
tiers = defaultdict(dict)
for _, row in hero_tier_list.iterrows():
    tiers[row["Name"]][row["Position"]] = row["Tier"]
 

In [110]:
# Make the graph
G = nx.DiGraph()

# Make a node for every hero
G.add_nodes_from(all_heroes)

# Add the position tier data
nx.set_node_attributes(G, tiers, name="tiers")

G.nodes["Frank"]

{'tiers': {'Mid': 'A', 'Top': 'S'}}

In [35]:
# Resolve tags to heros
def resolve_targets(raw: str, lookup: dict, all_heroes: set) -> list[str]:
    if not raw or raw.strip().lower() == "none":
        return []
    targets = []
    for token in raw.split(","):
        token = token.strip()
        if token in all_heroes:
            targets.append(token)
        elif token in lookup:
            targets.extend(lookup[token])  # tag -> heroes
    return list(set(targets))  # deduplicate

In [ ]:
# Edge columns
edge_types = {
    "synergies": "synergy",
    "counters": "counter",
    "countered_by": "countered_by",
    "anti-synergy": "anti_synergy",
}

for _, row in relationships.iterrows():
    hero = row["name"]
    for col, edge_type in edge_types.items():
        targets = resolve_targets(str(row[col]), lookup, all_heroes)
        for target in targets:
            if target == hero:
                continue
            G.add_edge(hero, target, type=edge_type)

In [53]:
G.out_edges("Kid", data=True)

OutEdgeDataView([('Kid', 'Mo', {'type': 'synergy'}), ('Kid', 'Manta', {'type': 'synergy'}), ('Kid', 'Lubos', {'type': 'synergy'}), ('Kid', 'Gillis', {'type': 'countered_by'}), ('Kid', 'Xiangxi Ke', {'type': 'synergy'}), ('Kid', 'Lan', {'type': 'synergy'}), ('Kid', 'Raven', {'type': 'synergy'}), ('Kid', 'Hass', {'type': 'countered_by'}), ('Kid', 'Qube', {'type': 'countered_by'}), ('Kid', 'Big Foot', {'type': 'countered_by'}), ('Kid', 'Cubey', {'type': 'countered_by'}), ('Kid', 'Peter', {'type': 'countered_by'}), ('Kid', 'Matata', {'type': 'countered_by'}), ('Kid', 'Frank', {'type': 'countered_by'}), ('Kid', 'Merisi', {'type': 'countered_by'}), ('Kid', 'Fatty White', {'type': 'countered_by'}), ('Kid', 'Bart', {'type': 'countered_by'}), ('Kid', 'Miki', {'type': 'countered_by'})])

In [ ]:
def score_hero(
        G, 
        candidate, 
        t1_available: set, t1_picked: set, 
        t2_available: set, t2_picked: set
    ):
    score = 0
    
    synergies = {v for _, v, d in G.out_edges(candidate, data=True) if d["type"] == "synergy"}
    counters = {v for _, v, d in G.out_edges(candidate, data=True) if d["type"] == "counter"}
    countered_by = {v for _, v, d in G.out_edges(candidate, data=True) if d["type"] == "countered_by"}
    anti_synergy = {v for _, v, d in G.out_edges(candidate, data=True) if d["type"] == "anti_synergy"}
    
    score += len(synergies & my_team)
    score += len(counters & enemy_team)
    score -= len(countered_by & enemy_team)  # enemy has something that beats me
    score -= len(anti_synergy & my_team)
    
    return score

In [114]:
# Simulate a draft 
t1_available = {
    "Top":     {"Frank", "Kid", "Wukong", "Hass", "Qube", "Cubey", "Reinhardt", "Merisi", "Foso", "Zealot"},
    "Mid":     {"Frank", "Wolfgang", "Merisi", "Foso", "Aurelio", "Crank", "Manta", "Raven", "Hass", "Qube", "Dylan"},
    "Jungler": {"Kamaitachi", "Aurelio", "Zealot", "Raven", "Manta", "Reinhardt", "Niels"},
    "Bot":     {"Felyn", "BaJie", "Foso", "Gang", "Niels", "Deep Space"},
    "Support": {"Acedia", "Blocker", "Peiniang Zhu", "Paisai", "Anna", "Dylan", "Peter", "Crank", "Lady Deadfire"}
}

t2_available = {
    "Top":     {"Frank", "Kid", "Wolfgang", "Hass", "Qube", "Mihawk", "Zealot", "Bart"},
    "Mid":     {"Frank", "Wolfgang", "Aurelio", "Manta", "Raven", "Hass", "Qube", "Lady Deadfire", "Lan"},
    "Jungler": {"Kamaitachi", "Aurelio", "Raven", "Manta", "Leon", "Lubos", "Gillis", "Bariel"},
    "Bot":     {"Felyn", "BaJie", "Quinn", "Ada", "Elemi", "Omaha", "Bond"},
    "Support": {"Acedia", "Blocker", "Beverly", "Tivie", "Charon", "Lady Deadfire", "Felicity"}
}

candidates = t1_available["Top"]
scores = {hero: score_hero(G, hero, {}, {}, position="Top") for hero in candidates}

TypeError: score_hero() got an unexpected keyword argument 'position'